# Credit Card Balance – Feature Engineering

Step 1 of preprocessing for `credit_card_balance.csv`: monthly statements of
the applicant's previous Home Credit credit cards. One row per card
(`SK_ID_PREV`) per month: balance, credit limit, withdrawals (ATM, shop,
other), payments, amounts owed and days past due (DPD = days late).

Needs `prev_loan_data_processed.parquet` from `Prevloans_data.ipynb` first,
for details of each card's original application.

**Sections**
1–2. Imports and settings
3. Load, keeping only applicants in the training data
4. Fill in missing values
5. Ratio features (utilisation, payments, withdrawals)
6. Per-card summaries, plus previous-application details, then per applicant
7. One row per applicant per month (cards added together)
8. Quarter-on-quarter changes
9. Monthly lateness flags
10. Combine the monthly tables
11. Summaries over recent periods (3 years … 3 months)
12. The latest month's values
13. Time until the first late payment
14. How regularly instalments were paid
15. Combine everything, plus recent-vs-long-term ratios
16. Add the `CREDITCARD_` prefix, quality checks
17. Save
18. Free memory

**Output:** `data/Model-Training/Staging/creditcard_processed.parquet` — one
row per applicant, columns prefixed `CREDITCARD_`.

Run by the `01_preprocessing` Airflow DAG (task `task_creditcard`, after
`task_prevloans`), or by hand.

## 1. Imports

In [73]:
import os
import pandas as pd
import numpy as np

## 2. Configuration & Paths

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

In [74]:
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_APP_TRAIN  = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_CC_BAL     = os.path.join(DATA_ROOT, "Input", "credit_card_balance.csv")
PATH_PREV_LOANS = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                               "prev_loan_data_processed.parquet")   # from Prevloans_data.ipynb
PATH_OUTPUT     = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                               "creditcard_processed.parquet")

## 3. Load Data

Only applicants in `application_train.csv` are kept. `TARGET` is read with
the IDs but dropped straight after the join, so it can't leak into the
features. The list of applicants is kept as the base for the final table.

In [75]:
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

CCBal = pd.read_csv(PATH_CC_BAL)
CCBal = CCBal.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})

# Oldest month first within each card, before filling gaps from neighbouring months
CCBal.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'],
    inplace=True, ignore_index=True,
)

# Keep only training applicants; drop TARGET straight away
CCBal = base.merge(CCBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

# Every applicant with credit card history -- the base the final table is built on (section 15)
unique_customers = CCBal[['SK_ID_CURR']].drop_duplicates()

print(f"CCBal shape   : {CCBal.shape}")
print(f"PrevLoans shape: {PrevLoans.shape}")

CCBal shape   : (3227965, 23)
PrevLoans shape: (1413701, 40)


Main amount columns (per card, per month):

- `AMT_BALANCE` — balance on the card
- `AMT_CREDIT_LIMIT_ACTUAL` — credit limit
- `AMT_DRAWINGS_CURRENT` — amount withdrawn/spent (split into `_ATM_`, `_POS_` (shops) and `_OTHER_`)
- `AMT_PAYMENT_CURRENT` — amount paid this month
- `AMT_PAYMENT_TOTAL_CURRENT` — total amount paid this month
- `AMT_RECEIVABLE_PRINCIPAL` — principal still owed (`AMT_TOTAL_RECEIVABLE` = total owed, including interest)

## 4. Fill in Missing Values

- **Withdrawal breakdowns:** if a month's total withdrawals is 0, its
  ATM/shop/other parts must be 0 too, so blanks there are set to 0.
- **Payments and minimum instalment:** a blank is filled from the nearest
  month of the same card (earlier months first), up to 6 months away.

In [76]:
# ── Withdrawal breakdowns: total is 0 -> each part is 0 ───────────────────
drawing_sub_amt = [
    'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT',
]
drawing_sub_cnt = [
    'CNT_DRAWINGS_ATM_CURRENT',
    'CNT_DRAWINGS_POS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT',
]
for col in drawing_sub_amt:
    CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & CCBal[col].isna(), col] = 0
for col in drawing_sub_cnt:
    CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & CCBal[col].isna(), col] = 0

# ── Payments: fill from the nearest month of the same card, up to 6 away ───
for col in ['AMT_PAYMENT_CURRENT', 'AMT_INST_MIN_REGULARITY']:
    CCBal[col] = (
        CCBal.groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=6).bfill(limit=6))
    )

print("Remaining nulls in imputed columns:")
check_cols = drawing_sub_amt + drawing_sub_cnt + ['AMT_PAYMENT_CURRENT', 'AMT_INST_MIN_REGULARITY']
print(CCBal[check_cols].isna().sum().to_string())

Remaining nulls in imputed columns:
AMT_DRAWINGS_ATM_CURRENT           0
AMT_DRAWINGS_OTHER_CURRENT         0
AMT_DRAWINGS_POS_CURRENT           0
CNT_DRAWINGS_ATM_CURRENT           0
CNT_DRAWINGS_POS_CURRENT           0
CNT_DRAWINGS_OTHER_CURRENT         0
AMT_PAYMENT_CURRENT           554904
AMT_INST_MIN_REGULARITY       170343


## 5. Ratio Features

All as percentages. Each denominator has 1 added so it can never be zero.

- `UTILIZATION_RATIO` — balance ÷ credit limit
- `PAYMENT_RATIO` — amount paid ÷ (amount paid + amount owed)
- `DRAWING_TO_PYMNT_RATIO` — amount withdrawn ÷ amount paid
- `INT_COVERAGE_RATIO` — interest owed (total owed − principal) ÷ amount paid
- `POS_` / `ATM_DRAWING_AMT_RATIO` — share of the withdrawn amount spent in shops / at ATMs
- `POS_` / `ATM_DRAWING_CNT_RATIO` — the same, by number of withdrawals

`compute_ratios()` is used again in section 7 on the monthly totals.
`features/credit_card.py` in the inference code repeats `UTILIZATION_RATIO`,
which the @prod model uses.

In [77]:
def compute_ratios(df):
    """Return a copy of df with the eight ratio columns added (percentages; +1 in each denominator avoids dividing by zero)."""
    df = df.copy()
    df['UTILIZATION_RATIO'] = np.where(
        df['AMT_CREDIT_LIMIT_ACTUAL'] >= 0,
        df['AMT_BALANCE'] / (1 + df['AMT_CREDIT_LIMIT_ACTUAL']) * 100,
        np.nan,
    )
    df['PAYMENT_RATIO'] = np.where(
        (df['AMT_PAYMENT_TOTAL_CURRENT'] + df['AMT_TOTAL_RECEIVABLE']) >= 0,
        df['AMT_PAYMENT_TOTAL_CURRENT']
        / (1 + df['AMT_PAYMENT_TOTAL_CURRENT'] + df['AMT_TOTAL_RECEIVABLE']) * 100,
        np.nan,
    )
    df['DRAWING_TO_PYMNT_RATIO'] = np.where(
        df['AMT_PAYMENT_TOTAL_CURRENT'] >= 0,
        df['AMT_DRAWINGS_CURRENT'] / (df['AMT_PAYMENT_TOTAL_CURRENT'] + 1) * 100,
        np.nan,
    )
    df['INT_COVERAGE_RATIO'] = np.where(
        df['AMT_PAYMENT_TOTAL_CURRENT'] >= 0,
        (df['AMT_TOTAL_RECEIVABLE'] - df['AMT_RECEIVABLE_PRINCIPAL'])
        / (df['AMT_PAYMENT_TOTAL_CURRENT'] + 1) * 100,
        np.nan,
    )
    df['POS_DRAWING_AMT_RATIO'] = np.where(
        df['AMT_DRAWINGS_CURRENT'] >= 0,
        df['AMT_DRAWINGS_POS_CURRENT'] / (1 + df['AMT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['ATM_DRAWING_AMT_RATIO'] = np.where(
        df['AMT_DRAWINGS_CURRENT'] >= 0,
        df['AMT_DRAWINGS_ATM_CURRENT'] / (1 + df['AMT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['POS_DRAWING_CNT_RATIO'] = np.where(
        df['CNT_DRAWINGS_CURRENT'] >= 0,
        df['CNT_DRAWINGS_POS_CURRENT'] / (1 + df['CNT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    df['ATM_DRAWING_CNT_RATIO'] = np.where(
        df['CNT_DRAWINGS_CURRENT'] >= 0,
        df['CNT_DRAWINGS_ATM_CURRENT'] / (1 + df['CNT_DRAWINGS_CURRENT']) * 100,
        np.nan,
    )
    return df


CCBal = compute_ratios(CCBal)
print("Ratio columns added:", [c for c in CCBal.columns if 'RATIO' in c])

Ratio columns added: ['UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO']


## 6. Per-Card Summaries, then per Applicant

1. One row per card (`SK_ID_PREV`): whether it was completed, its first and
   last credit limit, average balance, the average of each ratio, and how
   many months it was open.
2. Add each card's original application details from `prev_loan_data_processed.parquet`.
3. Per applicant: counts (cards, completed cards, insured, high/middle
   interest-rate groups, average decision date), then the max and mean of
   each per-card value across their cards.

In [78]:
# ── Step 1: one row per card ──────────────────────────────────────────────
CCBal_Prev_ID = (
    CCBal.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(
        # 1 if the card was ever marked Completed
        COMPLETED_PREV_LOAN=(
            'NAME_CONTRACT_STATUS',
            lambda x: int((x == 'Completed').any()),
        ),
        AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'first'),   # earliest month
        AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN=('AMT_CREDIT_LIMIT_ACTUAL', 'last'),       # latest month
        AMT_BALANCE_AVG_PREVLOAN=('AMT_BALANCE', 'mean'),
        AVG_UTILIZATION_RATIO_PREVLOAN=('UTILIZATION_RATIO', 'mean'),
        AVG_PAYMENT_RATIO_PREVLOAN=('PAYMENT_RATIO', 'mean'),
        AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN=('DRAWING_TO_PYMNT_RATIO', 'mean'),
        AVG_INT_COVERAGE_RATIO_PREVLOAN=('INT_COVERAGE_RATIO', 'mean'),
        AVG_POS_DRAWING_AMT_RATIO_PREVLOAN=('POS_DRAWING_AMT_RATIO', 'mean'),
        AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN=('ATM_DRAWING_AMT_RATIO', 'mean'),
        AVG_POS_DRAWING_CNT_RATIO_PREVLOAN=('POS_DRAWING_CNT_RATIO', 'mean'),
        AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN=('ATM_DRAWING_CNT_RATIO', 'mean'),
        DURATION_PREVLOAN=('MONTHS_BALANCE', lambda x: x.max() - x.min()),               # months open
    )
    .reset_index()
)

# ── Step 2: add each card's original application details ──────────────────
CCBal_Prev_ID = CCBal_Prev_ID.merge(
    PrevLoans, on=['SK_ID_CURR', 'SK_ID_PREV'], how='left'
)

# ── Step 3a: per-applicant counts ─────────────────────────────────────────
CCBal_Prev_ID_agg_1 = CCBal_Prev_ID.groupby('SK_ID_CURR').agg(
    TOTAL_PREV_LOAN_CNT=('SK_ID_PREV', 'nunique'),
    COMPLETED_PREV_LOAN_CNT=('COMPLETED_PREV_LOAN', 'sum'),
    NFLAG_INSURED_ON_APPROVAL_PREVLOAN_CNT=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    DAYS_DECISION_PREVLOAN_AVG=('DAYS_DECISION', 'mean'),
    HIGH_YIELD_GROUP_CNT=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
    MIDDLE_YIELD_GROUP_CNT=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
).reset_index()

# ── Step 3b: per-applicant max and mean of each per-card value ────────────
features_to_aggregate = [
    'AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN',
    'AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN',
    'AMT_BALANCE_AVG_PREVLOAN',
    'AVG_UTILIZATION_RATIO_PREVLOAN',
    'AVG_PAYMENT_RATIO_PREVLOAN',
    'AVG_DRAWING_TO_PYMNT_RATIO_PREVLOAN',
    'AVG_INT_COVERAGE_RATIO_PREVLOAN',
    'AVG_POS_DRAWING_AMT_RATIO_PREVLOAN',
    'AVG_ATM_DRAWING_AMT_RATIO_PREVLOAN',
    'AVG_POS_DRAWING_CNT_RATIO_PREVLOAN',
    'AVG_ATM_DRAWING_CNT_RATIO_PREVLOAN',
    'DURATION_PREVLOAN',
    'CREDIT_TO_APPL',
    'AMT_ANNUITY_TO_CREDIT_RATIO',
    'GOODSPRICE_TO_AMT',
]
CCBal_Prev_ID_agg_2 = (
    CCBal_Prev_ID
    .groupby('SK_ID_CURR')[features_to_aggregate]
    .agg(['max', 'mean'])
    .reset_index()   # make SK_ID_CURR a normal column again
)
# Two-level names like (column, 'max') -> COLUMN_MAX
CCBal_Prev_ID_agg_2.columns = [
    '_'.join(col).strip().upper() if col[1] else col[0].upper()
    for col in CCBal_Prev_ID_agg_2.columns
]

del CCBal_Prev_ID, PrevLoans, base
print(f"agg_1 shape: {CCBal_Prev_ID_agg_1.shape}")
print(f"agg_2 shape: {CCBal_Prev_ID_agg_2.shape}")

agg_1 shape: (86905, 7)
agg_2 shape: (86905, 31)


## 7. One Row per Applicant per Month

When an applicant had several cards in the same month, their amounts and
counts are **added together** into one row per applicant per month. The
ratios are then worked out again from those totals (rather than averaging
each card's ratio). `ISNULL_DRAWINGS_CURRENT_FLAG` = 1 for months with no withdrawals.

`features/credit_card.py` in the inference code builds the same monthly table.

In [79]:
sum_cols = [
    'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL',
    'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT',
    'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT',
    'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT',
    'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL',
    'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
    'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
    'CNT_INSTALMENT_MATURE_CUM',
]

# All of an applicant's cards added together, per month
Sum_CCBal_Mon = (
    CCBal.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[sum_cols]
    .sum()
    .reset_index()
)

# Ratios worked out again from the monthly totals
Sum_CCBal_Mon = compute_ratios(Sum_CCBal_Mon)
Sum_CCBal_Mon['ISNULL_DRAWINGS_CURRENT_FLAG'] = (
    Sum_CCBal_Mon['AMT_DRAWINGS_CURRENT'] == 0
).astype(int)

print(f"Sum_CCBal_Mon shape: {Sum_CCBal_Mon.shape}")

Sum_CCBal_Mon shape: (3222930, 28)


## 8. Quarter-on-Quarter (QoQ) Changes

Compares each month with the applicant's row 3 months earlier (3 rows back
in their monthly history):

- **% change** for amounts — negative amounts are set to 0 first; left
  blank when the earlier value is 0
- **Absolute change** for counts and ratios

An applicant's first 3 months are blank, as there's nothing to compare with.

In [80]:
def create_qoq_features(df, pct_cols, abs_cols):
    """
    Quarter-on-quarter change features.

    Parameters
    ----------
    df : DataFrame
        Must contain 'SK_ID_CURR', 'MONTHS_BALANCE' and every column in
        pct_cols and abs_cols, sorted by ['SK_ID_CURR', 'MONTHS_BALANCE'].
    pct_cols : list of str
        Amount columns -> <col>_QOQ_PCT_CHANGE (% change vs. 3 rows earlier).
        Negative values are set to 0 first, so sign flips don't give odd percentages.
    abs_cols : list of str
        Count/ratio columns -> <col>_QOQ_ABS_CHANGE (difference vs. 3 rows earlier).

    Returns
    -------
    DataFrame with SK_ID_CURR, MONTHS_BALANCE and only the new *_QOQ_* columns.
    """
    df = df.copy()
    grouped = df.groupby('SK_ID_CURR')

    df[pct_cols] = df[pct_cols].clip(lower=0)

    for col in pct_cols:
        prev_val = grouped[col].shift(3)
        df[f'{col}_QOQ_PCT_CHANGE'] = (
            (df[col] - prev_val)
            / prev_val.replace(0, np.nan)  # earlier value 0 -> blank, not infinity
        ).replace([np.inf, -np.inf], np.nan)

    for col in abs_cols:
        df[f'{col}_QOQ_ABS_CHANGE'] = df[col] - grouped[col].shift(3)

    qoq_cols = [c for c in df.columns if '_QOQ_' in c]
    return df[['SK_ID_CURR', 'MONTHS_BALANCE'] + qoq_cols]


# % change for these
amount_cols = [
    'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL',
    'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
    'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE'
]
# Absolute change for these
other_cols = [
    'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
    'CNT_INSTALMENT_MATURE_CUM',
    'UTILIZATION_RATIO', 'PAYMENT_RATIO',
    'DRAWING_TO_PYMNT_RATIO', 'INT_COVERAGE_RATIO',
    'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO',
    'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO'
]

QoQ_CCBal_Mon = create_qoq_features(Sum_CCBal_Mon, amount_cols, other_cols)
print(f"QoQ features shape: {QoQ_CCBal_Mon.shape}")

QoQ features shape: (3222930, 26)


## 9. Monthly Lateness Flags

Days past due can't be added across cards, so for each applicant and month
this takes the **highest** across their cards (`SK_DPD_MAX`,
`SK_DPD_DEF_MAX`), then flags months that were at least 1, 7 and 30 days
late (`DPD_GE_1`, `DPD_GE_7`, `DPD_GE_30`).

In [81]:
max_cols = [
    'SK_DPD', 'SK_DPD_DEF'
]

# Highest days-past-due across the applicant's cards, per month
Max_CCBal_Mon = (
    CCBal.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[['SK_DPD', 'SK_DPD_DEF']]
    .max()
    .reset_index()
)
# Not used: monthly max of the ratios as well
# ratio_max_cols = [
#     'UTILIZATION_RATIO', 'PAYMENT_RATIO', 'DRAWING_TO_PYMNT_RATIO',
#     'INT_COVERAGE_RATIO', 'POS_DRAWING_AMT_RATIO', 'ATM_DRAWING_AMT_RATIO',
#     'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO',
# ]
# ratio_max = (
#     Sum_CCBal_Mon.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[ratio_max_cols]
#     .max()
#     .reset_index()
# )
# Max_CCBal_Mon = Max_CCBal_Mon.merge(
#     ratio_max, on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left'
# )
Max_CCBal_Mon.columns = [
    col + '_MAX' if col not in ('SK_ID_CURR', 'MONTHS_BALANCE') else col
    for col in Max_CCBal_Mon.columns
]

# Late by at least 1 / 7 / 30 days that month
Max_CCBal_Mon['DPD_GE_1']  = (Max_CCBal_Mon['SK_DPD_MAX'] >= 1).astype(int)
Max_CCBal_Mon['DPD_GE_7']  = (Max_CCBal_Mon['SK_DPD_MAX'] >= 7).astype(int)
Max_CCBal_Mon['DPD_GE_30'] = (Max_CCBal_Mon['SK_DPD_MAX'] >= 30).astype(int)

print(f"Max_CCBal_Mon shape: {Max_CCBal_Mon.shape}")

Max_CCBal_Mon shape: (3222930, 7)


## 10. Combine the Monthly Tables

Totals and ratios (section 7), quarter-on-quarter changes (8) and lateness
flags (9) in one table, one row per applicant per month. `agg_cols` = every
column except the two keys; these are what get summarised below.

In [82]:
CCBal_Mon = (
    Sum_CCBal_Mon
    .merge(QoQ_CCBal_Mon,  on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left')
    .merge(Max_CCBal_Mon,  on=['SK_ID_CURR', 'MONTHS_BALANCE'], how='left')
)

agg_cols = [c for c in CCBal_Mon.columns if c not in ('SK_ID_CURR', 'MONTHS_BALANCE')]

print(f"CCBal_Mon shape: {CCBal_Mon.shape}")
print(f"Feature columns: {len(agg_cols)}")

CCBal_Mon shape: (3222930, 57)
Feature columns: 55


In [83]:
del Sum_CCBal_Mon, QoQ_CCBal_Mon, Max_CCBal_Mon
import gc; gc.collect()

0

## 11. Summaries over Recent Periods

The mean, max and (except for 3 months, where it isn't meaningful) standard
deviation of every monthly column, over:

| Table | Period | Suffix |
|---|---|---|
| `CCBal_3Y_Agg` | 3 years | `_LAST3Y` |
| `CCBal_2Y_Agg` | 2 years | `_LAST2Y` |
| `CCBal_1Y_Agg` | 1 year  | `_LAST1Y` |
| `CCBal_6M_Agg` | 6 months | `_LAST6M` |
| `CCBal_3M_Agg` | 3 months | `_LAST3M` |

Columns are named like `UTILIZATION_RATIO_MAX_LAST3M` (which the current
@prod model uses).

In [85]:
def build_window_agg(df, months_back, suffix, agg_funcs):
    """Apply agg_funcs to every column in agg_cols, per applicant, over the last `months_back` months (negative)."""
    window = df[df['MONTHS_BALANCE'] >= months_back]
    agg = (
        window.groupby('SK_ID_CURR')[agg_cols]
        .agg(agg_funcs)
        .reset_index()  # make SK_ID_CURR a normal column before renaming
    )
    # ('AMT_BALANCE', 'mean') -> 'AMT_BALANCE_MEAN_LAST3Y'
    agg.columns = [
        f"{col[0]}_{col[1]}{suffix}".upper() if col[1] else col[0]
        for col in agg.columns
    ]
    return agg


CCBal_3Y_Agg = build_window_agg(CCBal_Mon, -36, '_LAST3Y', ['mean','max','std'])
CCBal_2Y_Agg = build_window_agg(CCBal_Mon, -24, '_LAST2Y', ['mean','max','std'])
CCBal_1Y_Agg = build_window_agg(CCBal_Mon, -12, '_LAST1Y', ['mean','max','std'])
CCBal_6M_Agg = build_window_agg(CCBal_Mon,  -6, '_LAST6M', ['mean','max','std'])
CCBal_3M_Agg = build_window_agg(CCBal_Mon,  -3, '_LAST3M', ['mean','max'])

print("3Y:", CCBal_3Y_Agg.shape)
print("2Y:", CCBal_2Y_Agg.shape)
print("1Y:", CCBal_1Y_Agg.shape)
print("6M:", CCBal_6M_Agg.shape)
print("3M:", CCBal_3M_Agg.shape)

3Y: (86905, 166)
2Y: (86905, 166)
1Y: (86905, 166)
6M: (86905, 166)
3M: (85317, 111)


## 12. The Latest Month's Values

Each applicant's most recent month, with every column suffixed `_LAST_MONTH`.
It uses `.nth(-1)` (the actual last row) rather than `.last()`, which skips
blanks and could return an older month's value.

`DAYS_SINCE_LAST_TRANSACTION` = how many **months** ago that latest month was
(despite the name), as a positive number.

In [86]:
# Earlier version, kept for reference (not run)
#temp = CCBal_Mon.groupby('SK_ID_CURR')[['SK_ID_CURR','MONTHS_BALANCE'] + agg_cols ].nth(-1).reset_index()

In [87]:
# The actual latest month per applicant (.nth(-1); .last() would skip blanks)
CCBal_Last_Snapshot = (
    CCBal_Mon.sort_values(['SK_ID_CURR', 'MONTHS_BALANCE'])
    .groupby('SK_ID_CURR')[['SK_ID_CURR', 'MONTHS_BALANCE'] + agg_cols ]
    .nth(-1)
    .reset_index()
    .drop(columns=['index'])  # drop the old row number that reset_index() adds as a column
)
CCBal_Last_Snapshot.columns = [
    col + '_LAST_MONTH' if col != 'SK_ID_CURR' else col
    for col in CCBal_Last_Snapshot.columns
]
CCBal_Last_Snapshot.rename(
    columns={'MONTHS_BALANCE_LAST_MONTH': 'DAYS_SINCE_LAST_TRANSACTION'},
    inplace=True,
)
# MONTHS_BALANCE is negative; flip the sign to get "months ago"
CCBal_Last_Snapshot['DAYS_SINCE_LAST_TRANSACTION'] = (
    -CCBal_Last_Snapshot['DAYS_SINCE_LAST_TRANSACTION']
)

print(f"Last-snapshot shape: {CCBal_Last_Snapshot.shape}")

Last-snapshot shape: (86905, 57)


## 13. Time Until the First Late Payment

`DAYS_TO_DPD_GE1` = months from the applicant's first credit card month to
their first month at least 1 day late, × 30 to give approximate days. Blank
if never late.

In [88]:
# First month at least 1 day late, per applicant (months are oldest first)
CCBal_DPDge1 = (
    CCBal_Mon[CCBal_Mon['SK_DPD_MAX'] >= 1]
    .groupby('SK_ID_CURR')[['MONTHS_BALANCE']]
    .first()
    .reset_index()
    .rename(columns={'MONTHS_BALANCE': 'MONTH_DPD_GE1'})
)

# First month on record per applicant
CCBal_First_Snapshot = (
    CCBal_Mon.groupby('SK_ID_CURR')[['MONTHS_BALANCE']]
    .first()
    .reset_index()
)

CCBal_DPDge1 = CCBal_DPDge1.merge(
    CCBal_First_Snapshot, on='SK_ID_CURR', how='left'
)
# Months between the two (both negative, so the result is 0 or more), × 30 -> approx. days
CCBal_DPDge1['DAYS_TO_DPD_GE1'] = (
    (CCBal_DPDge1['MONTH_DPD_GE1'] - CCBal_DPDge1['MONTHS_BALANCE']) * 30
)
CCBal_DPDge1 = CCBal_DPDge1[['SK_ID_CURR', 'DAYS_TO_DPD_GE1']]

print(f"Customers with DPD ≥ 1 event: {len(CCBal_DPDge1):,}")

Customers with DPD ≥ 1 event: 17,882


## 14. How Regularly Instalments Were Paid

`INSTALLMENT_EVENT_RATE_AVG` = share of months in which the running count of
paid instalments (`CNT_INSTALMENT_MATURE_CUM`) went up, i.e. an instalment
was paid that month. Higher = more regular repayment.

In [89]:
CCBal_installment_feature = (
    CCBal_Mon.groupby('SK_ID_CURR')['CNT_INSTALMENT_MATURE_CUM']
    .apply(lambda x: (x.diff() > 0).mean())
    .reset_index(name='INSTALLMENT_EVENT_RATE_AVG')
)

del CCBal_Mon, CCBal
import gc; gc.collect()
print("Monthly tables released.")

Monthly tables released.


## 15. Combine Everything

Starts from the full list of applicants saved at load time and joins on
every summary — one row per applicant. For each recent period (3 years,
2 years, 1 year, 6 months, 3 months) a yes/no flag (`IS_IN_LAST_*`) says whether the
applicant had any card statement in it.

In [90]:
def flag_merge(left, right, on, flag_col):
    """Left-join `right` onto `left` and add `flag_col`: 1 if the row had a match in `right`, else 0."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


# Start from every applicant with credit card history (CCBal itself has been deleted)
final_cc_aggregated = unique_customers.copy()

# Per-card summaries (every applicant here has at least one card, so no flag needed)
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Prev_ID_agg_1, on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Prev_ID_agg_2, on='SK_ID_CURR', how='left')

# Recent-period summaries, each with a "had statements in this period" flag
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_3Y_Agg, 'SK_ID_CURR', 'IS_IN_LAST_3Y')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_2Y_Agg, 'SK_ID_CURR', 'IS_IN_LAST_2Y')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_1Y_Agg, 'SK_ID_CURR', 'IS_IN_LAST_1Y')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_6M_Agg, 'SK_ID_CURR', 'IS_IN_LAST_6M')
final_cc_aggregated = flag_merge(final_cc_aggregated, CCBal_3M_Agg, 'SK_ID_CURR', 'IS_IN_LAST_3M')

# Latest month, time to first late payment, repayment regularity
final_cc_aggregated = final_cc_aggregated.merge(CCBal_Last_Snapshot,       on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_DPDge1,              on='SK_ID_CURR', how='left')
final_cc_aggregated = final_cc_aggregated.merge(CCBal_installment_feature, on='SK_ID_CURR', how='left')

print(f"Final shape: {final_cc_aggregated.shape}")
final_cc_aggregated.head(3)

Final shape: (86905, 704)


,SK_ID_CURR,TOTAL_PREV_LOAN_CNT,COMPLETED_PREV_LOAN_CNT,NFLAG_INSURED_ON_APPROVAL_PREVLOAN_CNT,DAYS_DECISION_PREVLOAN_AVG,HIGH_YIELD_GROUP_CNT,MIDDLE_YIELD_GROUP_CNT,AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN_MAX,AMT_CREDIT_LIMIT_ACTUAL_ORIGINAL_PREVLOAN_MEAN,AMT_CREDIT_LIMIT_ACTUAL_FINAL_PREVLOAN_MAX,...,ATM_DRAWING_AMT_RATIO_QOQ_ABS_CHANGE_LAST_MONTH,POS_DRAWING_CNT_RATIO_QOQ_ABS_CHANGE_LAST_MONTH,ATM_DRAWING_CNT_RATIO_QOQ_ABS_CHANGE_LAST_MONTH,SK_DPD_MAX_LAST_MONTH,SK_DPD_DEF_MAX_LAST_MONTH,DPD_GE_1_LAST_MONTH,DPD_GE_7_LAST_MONTH,DPD_GE_30_LAST_MONTH,DAYS_TO_DPD_GE1,INSTALLMENT_EVENT_RATE_AVG
0,100006,1,0,0.0,-181.0,0,0,270000,270000.0,270000,...,0.0,0.0,0.0,0,0,0,0,0,NaN,0.000000
1,100011,1,0,0.0,-2279.0,0,0,180000,180000.0,90000,...,0.0,0.0,0.0,0,0,0,0,0,NaN,0.445946
2,100021,1,1,0.0,-549.0,0,0,675000,675000.0,675000,...,0.0,0.0,0.0,0,0,0,0,0,NaN,0.000000


**Recent vs. long-term ratios:** each listed feature's last-6-months average
÷ its last-3-years average. Above 1 means the value has risen recently —
e.g. for lateness, the applicant has been late more often lately. Left
blank when the 3-year value is 0.

In [91]:
def add_period_ratio_features(df, attributes, numerator_period='6M', denominator_period='3Y'):
    """
    For each attribute, add <attr>_LAST<numerator_period>_TO_<denominator_period>_RATIO
    = <attr>_LAST<numerator_period> ÷ <attr>_LAST<denominator_period>,
    left blank (NaN) when the denominator is 0.

    Both <attr>_LAST... columns must already exist in df (here they come
    from build_window_agg in section 11).
    """
    for attr in attributes:
        numerator_col = f'{attr}_LAST{numerator_period}'
        denominator_col = f'{attr}_LAST{denominator_period}'
        ratio_col = f'{attr}_LAST{numerator_period}_TO_{denominator_period}_RATIO'
        df[ratio_col] = np.where(
            df[denominator_col] != 0,
            df[numerator_col] / df[denominator_col],
            np.nan,
        )
    return df


recent_vs_historical_attrs = [
    'DPD_GE_1_MEAN',
    'ATM_DRAWING_CNT_RATIO_MEAN',
    'POS_DRAWING_AMT_RATIO_MEAN',
    'INT_COVERAGE_RATIO_MEAN',
    'DRAWING_TO_PYMNT_RATIO_MEAN',
    'PAYMENT_RATIO_MEAN',
    'UTILIZATION_RATIO_MEAN',
    'SK_DPD_MAX_MEAN',
    'AMT_TOTAL_RECEIVABLE_MEAN',
    'AMT_RECIVABLE_MEAN',
    'AMT_PAYMENT_CURRENT_MEAN',
    'AMT_PAYMENT_TOTAL_CURRENT_MEAN',
    'AMT_INST_MIN_REGULARITY_MEAN',
    'AMT_DRAWINGS_POS_CURRENT_MEAN',
    'AMT_DRAWINGS_CURRENT_MEAN',
    'AMT_CREDIT_LIMIT_ACTUAL_MEAN',
    'AMT_BALANCE_MEAN',
]

final_cc_aggregated = add_period_ratio_features(
    final_cc_aggregated, recent_vs_historical_attrs,
    numerator_period='6M', denominator_period='3Y',
)

## 16. Add the Prefix, Quality Checks

Every feature gets the `CREDITCARD_` prefix. Then: the share of missing
values in the emptiest columns, and exactly one row per applicant.

In [92]:
# Prefix all feature columns with CREDITCARD_
final_cc_aggregated.columns = [
    'CREDITCARD_' + col if col != 'SK_ID_CURR' else col
    for col in final_cc_aggregated.columns
]

# Null rates
null_rates = final_cc_aggregated.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

# No duplicate customers
n_dupes = final_cc_aggregated['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR. Final row count: {len(final_cc_aggregated):,}")

Null rate (top 15 columns):
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_LAST_MONTH     0.998585
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST6M     0.998147
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST1Y     0.996157
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST3M    0.995639
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST3M     0.995639
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_STD_LAST3Y     0.991911
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST6M     0.991174
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST6M    0.991174
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST1Y    0.984190
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST1Y     0.984190
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MEAN_LAST3Y    0.970566
CREDITCARD_AMT_DRAWINGS_OTHER_CURRENT_QOQ_PCT_CHANGE_MAX_LAST3Y     0.970566
CREDITCARD_DPD_GE_1_MEAN_LAST6M_TO_3Y_RATIO     

## 17. Save

In [ ]:
final_cc_aggregated.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\Staging\creditcard_processed.parquet


: 

## 18. Free Memory

In [72]:
del (
    CCBal_installment_feature, CCBal_DPDge1, CCBal_Last_Snapshot,
    CCBal_3M_Agg, CCBal_6M_Agg, CCBal_1Y_Agg, CCBal_2Y_Agg, CCBal_3Y_Agg,
    CCBal_Prev_ID_agg_2, CCBal_Prev_ID_agg_1,
    unique_customers,
)
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
